# 03 SQL Report (B)

Bu notebook-da gold cədvəllər üzərində SQL sorğuları ilə biznes suallarına cavab veririk:
- B1: Gec çatdırılma rəyi pisləşdirirmi?
- B2: Hansı satıcılar daha yaxşı və ya pis performans göstərir?
- B3: Hansı məhsul kateqoriyaları gəlir və məmnuniyyət gətirir?
- B4: Yağış sifarişlərə təsir edirmi? (yalnız São Paulo)
- B5: Bayramlar satışı dəyişirmi?


In [1]:
import sys
from pathlib import Path
import pandas as pd

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
sys.path.append(str(_root / "notebooks"))
from db import get_engine

engine = get_engine()

## B1: Gec çatdırılma rəyi pisləşdirirmi?

Vaxtında və gec çatan sifarişlərin orta rəy balını və narazı rəy  faizini müqayisə edirik. Gecikməni təqvim günü ilə hesablayırıq: real çatdırılma günü gözlənilən gündən sonradırsa, sifariş gecdir.

Vaxtında çatan sifarişlərin orta balı 4,29, narazı rəy faizi 9,28%-dir. Gec çatanlarda orta bal 2,27, narazı rəy faizi isə 62,41%-dir. Gecikmə müştəri məmnuniyyətinə güclü təsir edir.

In [2]:
df = pd.read_sql("""
    SELECT
        CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) > 0
             THEN 'Gecikib' ELSE 'Vaxtinda' END AS delivery_status,
        COUNT(*) AS total_reviews,
        ROUND(AVG(CAST(r.review_score AS FLOAT)), 2) AS avg_review_score,
        ROUND(AVG(CASE WHEN r.review_score <= 2 THEN 1.0 ELSE 0.0 END) * 100, 2) AS unhappy_review_pct
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) > 0
                  THEN 'Gecikib' ELSE 'Vaxtinda' END
""", engine)

df

,delivery_status,total_reviews,avg_review_score,unhappy_review_pct
0,Vaxtinda,89944,4.29,9.28
1,Gecikib,6409,2.27,62.41


In [3]:
df.to_sql("report_late_vs_review", engine, schema="gold", if_exists="replace", index=False)

-1

Gecikmə müddəti artdıqca narazı rəy faizinin necə dəyişdiyinə baxırıq.
Vaxtında çatanlarda narazı rəy 9,28%, 1-3 gün gecikəndə 32,17%, 4-7 gün gecikəndə 67,65%, 8+ gün gecikəndə 79,19%-dir. Gecikmə uzandıqca narazılıq kəskin artır.

In [4]:
df = pd.read_sql("""
    SELECT
        delay_bucket,
        COUNT(*) AS total_reviews,
        ROUND(AVG(CASE WHEN review_score <= 2 THEN 1.0 ELSE 0.0 END) * 100, 2) AS unhappy_review_pct
    FROM (
        SELECT
            r.review_score,
            CASE
                WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) <= 0 THEN 'Vaxtinda'
                WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) <= 3 THEN '1-3 gun gecikib'
                WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) <= 7 THEN '4-7 gun gecikib'
                ELSE '8+ gun gecikib'
            END AS delay_bucket
        FROM silver.orders o
        INNER JOIN gold.fact_reviews r ON o.order_id = r.order_id
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date IS NOT NULL
          AND o.order_estimated_delivery_date IS NOT NULL
    ) AS t
    GROUP BY delay_bucket
    ORDER BY unhappy_review_pct
""", engine)

df

,delay_bucket,total_reviews,unhappy_review_pct
0,Vaxtinda,89944,9.28
1,1-3 gun gecikib,1856,32.17
2,4-7 gun gecikib,1756,67.65
3,8+ gun gecikib,2797,79.19


In [5]:
df.to_sql("report_delay_buckets", engine, schema="gold", if_exists="replace", index=False)

-1

## B2 Hansı satıcılar ən çox satış edir və daşınma xərci necədir?

Ən çox satış edən 10 satıcıya baxırıq və onların sifariş başına orta satışını, orta daşınma haqqını və daşınmanın satışa nisbətini müqayisə edirik.
1-ci satıcı 226 987,93 satışla birincidir, daşınma xərci satışın 8,82%-ni təşkil edir. 5-ci satıcıda sifariş başına orta daşınma haqqı (52,66) və daşınmanın satışa nisbəti (27,46%) ilk 10 arasında ən yüksəkdir. 6-cı satıcıda isə daşınma payı ən aşağıdır (3,61%). Yəni satışı yüksək olan satıcılarda daşınma xərci çox fərqlənə bilər.

In [6]:
df = pd.read_sql("""
    SELECT TOP 10
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        ROUND(CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0), 2) AS avg_sales_per_order,
        ROUND(CAST(total_freight AS DECIMAL(18, 2)) / NULLIF(total_orders, 0), 2) AS avg_freight_per_order,
        ROUND(CAST(total_freight AS DECIMAL(18, 2)) * 100 / NULLIF(total_sales, 0), 2) AS freight_to_sales_pct
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df

,seller_id,total_orders,total_sales,total_freight,avg_sales_per_order,avg_freight_per_order,freight_to_sales_pct
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,201.95,17.81,8.82
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,12856.58,626.27,36.94,5.90
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,111.11,19.38,17.44
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,9916.36,330.31,17.16,5.19
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,51236.64,191.75,52.66,27.46
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,5992.06,520.32,18.78,3.61
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,121.90,18.99,15.57
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,121.97,18.01,14.76
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,33716.40,151.88,37.05,24.40
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,104.55,19.64,18.79


In [7]:
df.to_sql("report_top_sellers_by_sales", engine, schema="gold", if_exists="replace", index=False)

-1

Sifariş başına orta satışa görə ilk 10 satıcıya baxırıq.
İlk yerlərdə yalnız 1-5 sifarişi olan satıcılar var. Məsələn, birinci satıcının cəmi 1 sifarişi var və orta satışı 6735-dir. Bu, az sifarişli satıcıların orta göstəricisinin yanıldıcı ola biləcəyini göstərir.

In [8]:
df = pd.read_sql("""
    SELECT TOP 10
        seller_id,
        total_orders,
        total_sales,
        ROUND(CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0), 2) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY avg_sales_per_order DESC
""", engine)

df

,seller_id,total_orders,total_sales,avg_sales_per_order
0,e3b4998c7a498169dc7bce44e6bb6277,1,6735.00,6735.00
1,80ceebb4ee9b31afb6c6a916a574a1e2,1,6729.00,6729.00
2,ee27a8f15b1dded4d213a468ba4eb391,1,6499.00,6499.00
3,b37c4c02bda3161a7546a4e6d222d5b2,4,24075.00,6018.75
4,585175ec331ea177fa47199e39a6170a,1,3549.00,3549.00
5,abe021b01ba992245271b9aa422032df,2,6720.00,3360.00
6,a00824eb9093d40e589b940ec45c4eb0,3,9399.97,3133.32
7,e2a1ac9bf33e5549a2a4f834e70df2f8,5,14999.45,2999.89
8,e908c0f3646e8b60375734a350d95d71,1,2951.00,2951.00
9,d63c73efd41eb002280e7ec831424edb,2,5598.00,2799.00


In [9]:
df.to_sql("report_sellers_avg_sales", engine, schema="gold", if_exists="replace", index=False)

-1

Ən çox sifarişi olan 10 satıcıya baxırıq və onların satış və daşınma göstəricilərini müqayisə edirik.
Ən çox sifarişi olan satıcının 1 819 sifarişi var, amma sifariş başına orta satışı cəmi 66,36-dır, yəni çox sifariş çox satış demək deyil. Bəzi satıcılarda daşınma xərci satışa nisbətən çox yüksəkdir: 8-ci satıcının 1 132 sifarişi var, amma satışı cəmi 36 696,76-dır və daşınma payı 47,12%-dir. 4-cü satıcıda bu pay 32,87%-dir. Satışa görə ilk 10-da olan 4869f7a5... satıcısı isə sifariş sayına görə də ilk 10-dadır, daşınma payı yalnız 8,82%-dir.

In [10]:
df = pd.read_sql("""
    SELECT TOP 10
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        ROUND(CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0), 2) AS avg_sales_per_order,
        ROUND(CAST(total_freight AS DECIMAL(18, 2)) / NULLIF(total_orders, 0), 2) AS avg_freight_per_order,
        ROUND(CAST(total_freight AS DECIMAL(18, 2)) * 100 / NULLIF(total_sales, 0), 2) AS freight_to_sales_pct
    FROM gold.mart_seller_performance
    ORDER BY total_orders DESC
""", engine)

df

,seller_id,total_orders,total_sales,total_freight,avg_sales_per_order,avg_freight_per_order,freight_to_sales_pct
0,6560211a19b47992c3666cc44a7e94c0,1819,120702.83,27348.11,66.36,15.03,22.66
1,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,111.11,19.38,17.44
2,cc419e0650a3c5ba77189a1882b7556a,1651,101090.96,24846.03,61.23,15.05,24.58
3,1f50f920176fa81dab994f9023523100,1399,106655.71,35057.23,76.24,25.06,32.87
4,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,121.90,18.99,15.57
5,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,104.55,19.64,18.79
6,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,121.97,18.01,14.76
7,ea8482cd71df3c1969d7b9473ff13abc,1132,36696.76,17291.87,32.42,15.28,47.12
8,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,201.95,17.81,8.82
9,3d871de0142ce09b7081e2b9d1733cb1,1064,93407.40,22108.25,87.79,20.78,23.67


In [11]:
df.to_sql("report_top_sellers_by_orders", engine, schema="gold", if_exists="replace", index=False)

-1

Ən azı 100 sifarişi olan satıcılar arasında sifariş başına orta satışa görə ilk 10-a baxırıq. Beləcə az sifarişli satıcıların yanıldıcı göstəricisini kənarlaşdırırıq.
Birinci satıcının sifariş başına orta satışı ən yüksəkdir 626,27 və 348 sifarişi var. 2-ci satıcının orta satışı 520,32-dir 319 sifariş. Bu, az sifarişli satıcıların yüksək göstəricilərinin təsirini azaldır, amma yüksək orta satışın satıcının daha çox mənfəət qazandığını sübut etmir.

In [12]:
df = pd.read_sql("""
    SELECT TOP 10
        seller_id,
        total_orders,
        total_sales,
        ROUND(CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0), 2) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    WHERE total_orders >= 100
    ORDER BY avg_sales_per_order DESC
""", engine)

df

,seller_id,total_orders,total_sales,avg_sales_per_order
0,53243585a1d6dc2643021fd1853d8905,348,217940.44,626.27
1,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,520.32
2,edb1ef5e36e0c8cd84eb3c9b003e486d,165,79144.90,479.67
3,f5a590cf36251cf1162ea35bef76fe84,113,51232.78,453.39
4,ccc4bbb5f32a6ab2b7066a4130f114e3,184,72926.72,396.34
5,25c5c91f63607446a97b143d2d535d31,156,54208.78,347.49
6,5dceca129747e92ff8ef7a997dc4f8ca,322,111126.73,345.11
7,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,330.31
8,522620dcb18a6b31cd7bdf73665113a9,172,56690.49,329.60
9,17a053fcb14bd219540cbde0df490be0,124,40353.24,325.43


In [13]:
df.to_sql("report_sellers_100_plus", engine, schema="gold", if_exists="replace", index=False)

-1

Ən çox satış edən satıcı 226987,93 satışla birincidir. Ən çox sifarişi olan satıcının 1819 sifariş isə sifariş başına orta satışı cəmi 66,36-dır, yəni çox sifariş çox satış demək deyil. Daşınma xərcinin satışdakı payı satıcılar arasında çox fərqlənir: 3,61%-dən 47,12%-ə qədər. Satıcıları düzgün müqayisə etmək üçün az sifarişli satıcıları kənarlaşdırmaq lazımdır

## B3. Hansı məhsul kateqoriyaları gəlir və yüksək rəy gətirir?

Gəlirinə görə ilk 10 məhsul kateqoriyasına baxırıq və hər kateqoriyada satılan məhsul sayını və orta qiyməti müqayisə edirik.
health_beauty kateqoriyası 1,26 milyon gəlirlə birincidir. bed_bath_table 11 115 məhsulla ən çox satılan kateqoriyadır, amma orta qiyməti aşağıdır. watches_gifts isə orta qiymətin ən yüksək olduğu kateqoriyadır 201,14 və gəlirə görə 2-ci yerdədir. Gəlir həm satılan məhsul sayından, həm də orta qiymətdən asılıdır.

In [14]:
df = pd.read_sql("""
    SELECT TOP 10
        p.category_name_english,
        SUM(oi.price) AS total_revenue,
        COUNT(*) AS items_sold,
        ROUND(AVG(CAST(oi.price AS FLOAT)), 2) AS avg_price
    FROM silver.order_items oi
    INNER JOIN gold.dim_products p
        ON oi.product_id = p.product_id
    GROUP BY p.category_name_english
    ORDER BY total_revenue DESC
""", engine)

df

,category_name_english,total_revenue,items_sold,avg_price
0,health_beauty,1258681.34,9670,130.16
1,watches_gifts,1205005.68,5991,201.14
2,bed_bath_table,1036988.68,11115,93.30
3,sports_leisure,988048.97,8641,114.34
4,computers_accessories,911954.32,7827,116.51
5,furniture_decor,729762.49,8334,87.56
6,cool_stuff,635290.85,3796,167.36
7,housewares,632248.66,6964,90.79
8,auto,592720.11,4235,139.96
9,garden_tools,485256.46,4347,111.63


In [15]:
df.to_sql("report_category_revenue", engine, schema="gold", if_exists="replace", index=False)

-1

Kateqoriyalar üzrə orta rəy balına görə ilk 10-a baxırıq.
 İlk yerlərdə cds_dvds_musicals 4,64 və fashion_childrens_clothes 4,50 var, amma onların cəmi 14 və 8 rəyi var. Bu qədər az rəylə orta bal etibarlı deyil. books_general_interest kateqoriyasının isə 549 rəyi var və orta bal 4,45-dir. Yəni az rəyli kateqoriyalar siyahını yanıldır, ona görə növbəti sorğuda yalnız 50-dən çox rəyi olan kateqoriyalara baxırıq.

In [16]:
df = pd.read_sql("""
    SELECT TOP 10
        p.category_name_english,
        COUNT(r.review_score) AS total_reviews,
        ROUND(AVG(CAST(r.review_score AS FLOAT)), 2) AS avg_review_score
    FROM silver.order_items oi
    INNER JOIN gold.dim_products p
        ON oi.product_id = p.product_id
    INNER JOIN gold.fact_reviews r
        ON oi.order_id = r.order_id
    GROUP BY p.category_name_english
    ORDER BY avg_review_score DESC
""", engine)

df

,category_name_english,total_reviews,avg_review_score
0,cds_dvds_musicals,14,4.64
1,fashion_childrens_clothes,8,4.50
2,books_general_interest,549,4.45
3,costruction_tools_tools,99,4.44
4,flowers,31,4.42
5,books_imported,60,4.40
6,books_technical,266,4.37
7,food_drink,279,4.32
8,luggage_accessories,1088,4.32
9,small_appliances_home_oven_and_coffee,76,4.30


In [17]:
df.to_sql("report_category_reviews_all", engine, schema="gold", if_exists="replace", index=False)

-1

Ən azı 50 rəyi olan kateqoriyalar arasında müştərilərin ən razı qaldığı 10 kateqoriyanı tapmaq. Rəyi çox az olan kateqoriyalar təsadüfi yüksək bal ala bilər, ona görə > 50 şərti qoyuldu.
Ən yüksək orta bal books_general_interest kateqoriyasındadır 4.46, 508 rəy. Ardınca costruction_tools_tools 4.43 və books_technical 4.41 gəlir. İlk 10 kateqoriyanın hamısında orta bal 4.24-dən yüksəkdir. Ən çox rəyi olan kateqoriya pet_shop-dur.

In [18]:
df = pd.read_sql("""
    WITH order_categories AS (
        SELECT DISTINCT
            oi.order_id,
            p.category_name_english
        FROM silver.order_items oi
        INNER JOIN gold.dim_products p
            ON oi.product_id = p.product_id
    )
    SELECT TOP 10
        oc.category_name_english,
        COUNT(DISTINCT r.review_id) AS total_reviews,
        ROUND(AVG(CAST(r.review_score AS FLOAT)), 2) AS avg_review_score
    FROM order_categories oc
    INNER JOIN gold.fact_reviews r
        ON oc.order_id = r.order_id
    GROUP BY oc.category_name_english
    HAVING COUNT(DISTINCT r.review_id) > 50
    ORDER BY avg_review_score DESC
""", engine)

df

,category_name_english,total_reviews,avg_review_score
0,books_general_interest,508,4.46
1,costruction_tools_tools,94,4.43
2,books_technical,259,4.41
3,food_drink,226,4.39
4,luggage_accessories,1030,4.33
5,books_imported,53,4.32
6,small_appliances_home_oven_and_coffee,75,4.29
7,food,445,4.28
8,cine_photo,66,4.24
9,pet_shop,1700,4.24


In [19]:
df.to_sql("report_category_reviews_50plus", engine, schema="gold", if_exists="replace", index=False)

-1

Ən çox gəlir gətirən kateqoriyaların müştəri məmnuniyyətini orta rəy balı ilə birlikdə müqayisə etmək. Yalnız 50-dən çox rəyi olan kateqoriyalar daxil edilib.
Ən çox gəliri health_beauty gətirir . İkinci watches_gifts , üçüncü bed_bath_table . Maraqlısı budur ki, ən çox satış olan bed_bath_table 11,115 məhsul ən aşağı rəy balına malikdir 3.97. sports_leisure və cool_stuff isə yüksək gəlirlə yanaşı yaxşı bal alıb 4.17.

In [20]:
df = pd.read_sql("""
    WITH category_sales AS (
        SELECT
            p.category_name_english,
            SUM(oi.price) AS total_revenue,
            COUNT(*) AS items_sold
        FROM silver.order_items oi
        INNER JOIN gold.dim_products p
            ON oi.product_id = p.product_id
        GROUP BY p.category_name_english
    ),
    order_categories AS (
        SELECT DISTINCT
            oi.order_id,
            p.category_name_english
        FROM silver.order_items oi
        INNER JOIN gold.dim_products p
            ON oi.product_id = p.product_id
    ),
    category_reviews AS (
        SELECT
            oc.category_name_english,
            COUNT(DISTINCT r.review_id) AS total_reviews,
            AVG(CAST(r.review_score AS FLOAT)) AS avg_review_score
        FROM order_categories oc
        INNER JOIN gold.fact_reviews r
            ON oc.order_id = r.order_id
        GROUP BY oc.category_name_english
        HAVING COUNT(DISTINCT r.review_id) > 50
    )
    SELECT TOP 10
        s.category_name_english,
        ROUND(s.total_revenue, 2) AS total_revenue,
        s.items_sold,
        r.total_reviews,
        ROUND(r.avg_review_score, 2) AS avg_review_score
    FROM category_sales s
    INNER JOIN category_reviews r
        ON s.category_name_english = r.category_name_english
    ORDER BY s.total_revenue DESC
""", engine)

df

,category_name_english,total_revenue,items_sold,total_reviews,avg_review_score
0,health_beauty,1258681.34,9670,8767,4.18
1,watches_gifts,1205005.68,5991,5582,4.07
2,bed_bath_table,1036988.68,11115,9324,3.97
3,sports_leisure,988048.97,8641,7656,4.17
4,computers_accessories,911954.32,7827,6641,4.03
5,furniture_decor,729762.49,8334,6404,4.01
6,cool_stuff,635290.85,3796,3608,4.17
7,housewares,632248.66,6964,5854,4.14
8,auto,592720.11,4235,3879,4.09
9,garden_tools,485256.46,4347,3505,4.13


In [21]:
df.to_sql("report_category_revenue_vs_reviews", engine, schema="gold", if_exists="replace", index=False)

-1

Satış və müştəri rəylərinin müqayisəsi göstərdi ki, health_beauty kateqoriyası ən yüksək gəlir əldə edib. bed_bath_table kateqoriyası ən çox satılan məhsul sayına malik olsa da, orta rəy balı digər bəzi yüksək gəlirli kateqoriyalardan aşağıdır. sports_leisure kateqoriyası isə yüksək satış gəlirini yaxşı müştəri rəyi ilə birləşdirir. Bu nəticələr satış gəliri ilə müştəri məmnuniyyətinin kateqoriyalar arasında fərqləndiyini göstərir.

## B4: Yağış sifarişlərə təsir edirmi?


Sao Paulo (SP) ştatında yağışlı və yağışsız günlərdə sifariş sayı və satış dəyişirmi?
Günləri Rainy yağış > 0 və Dry yağışsız kimi ayırıb orta gündəlik sifariş sayını və orta gündəlik satışı müqayisə edirik.
Yağışlı günlərdə 682 gün orta sifariş sayı 91.83, yağışsız günlərdə 414 gün isə 81.77 olub. Orta gündəlik satış da yağışlı günlərdə daha yüksəkdir 12,501.53 qarşı 11,341.67. Bu, yağış ilə sifariş və satış arasında əlaqə olduğunu göstərir, amma səbəb-nəticə əlaqəsini təsdiqləmir.

In [22]:
df = pd.read_sql("""
    SELECT
        CASE WHEN sp_precipitation > 0 THEN 'Rainy' ELSE 'Dry' END AS weather_type,
        COUNT(*) AS total_days,
        ROUND(AVG(CAST(total_orders AS FLOAT)), 2) AS avg_orders,
        ROUND(AVG(CAST(total_sales AS FLOAT)), 2) AS avg_sales
    FROM gold.mart_daily_sales_weather
    GROUP BY CASE WHEN sp_precipitation > 0 THEN 'Rainy' ELSE 'Dry' END
""", engine)

df

,weather_type,total_days,avg_orders,avg_sales
0,Dry,414,81.77,11341.67
1,Rainy,682,91.83,12501.53


In [23]:
df.to_sql("report_rain_orders_sales", engine, schema="gold", if_exists="replace", index=False)

-1

SP ştatında yağışlı günlərdə gecikmiş çatdırılmaların payı artır?
Hər sifarişi alındığı günün havasına görə Rainy və Dry qruplarına bölmək. Faktiki çatdırılma tarixi təxmini tarixdən günə görə gec olarsa, sifariş gecikmiş sayılır. Hər qrup üçün sifariş sayını, gecikmiş sifariş sayını və gecikmə faizini hesablamaq.
Yağışsız günlərdə sifarişlərin 4.24%-i , yağışlı günlərdə isə 8.14%-i gecikib. Yəni yağışlı günlərdə gecikmə faizi təxminən 3.9 faiz bəndi, yəni təxminən 2 dəfə yüksəkdir. Bu, yağışla gecikmə arasında əlaqə olduğunu göstərir.

In [24]:
df = pd.read_sql("""
    SELECT
        CASE WHEN w.sp_precipitation > 0 THEN 'Rainy' ELSE 'Dry' END AS weather_type,
        COUNT(*) AS total_orders,
        SUM(CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) > 0
                 THEN 1 ELSE 0 END) AS late_orders,
        ROUND(AVG(CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) > 0
                       THEN 1.0 ELSE 0.0 END) * 100, 2) AS late_order_pct
    FROM silver.orders o
    INNER JOIN gold.mart_daily_sales_weather w
        ON CAST(o.order_purchase_timestamp AS DATE) = w.sales_date
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY CASE WHEN w.sp_precipitation > 0 THEN 'Rainy' ELSE 'Dry' END
""", engine)

df

,weather_type,total_orders,late_orders,late_order_pct
0,Dry,33846,1435,4.24
1,Rainy,62624,5099,8.14


In [25]:
df.to_sql("report_rain_late_orders", engine, schema="gold", if_exists="replace", index=False)

-1

## B5: Bayramlar satışı dəyişirmi?

Bayram günlərində satış adi günlərdən fərqlənirmi?
Günləri Holiday  və Normal day qruplarına bölüb hər qrupda günlərin sayını, orta gündəlik sifariş sayını və orta gündəlik satışı müqayisə edirik.

Bayram günlərində 39 gün orta sifariş sayı 63.72, adi günlərdə 1057 gün isə 88.92 olub, yəni təxminən 28% azdır. Orta gündəlik satış da bayram günlərində 842909, adi günlərdə 1219751-dir, yəni təxminən 31% aşağıdır. Seçilmiş dövrdə bayram günlərində satış aktivliyi daha aşağı olub.

In [26]:
df = pd.read_sql("""
    SELECT
        CASE WHEN is_holiday = 1 THEN 'Holiday' ELSE 'Normal day' END AS day_type,
        COUNT(*) AS total_days,
        ROUND(AVG(CAST(total_orders AS FLOAT)), 2) AS avg_orders,
        ROUND(AVG(CAST(total_sales AS FLOAT)), 2) AS avg_sales
    FROM gold.mart_daily_sales_weather
    GROUP BY CASE WHEN is_holiday = 1 THEN 'Holiday' ELSE 'Normal day' END
""", engine)

df

,day_type,total_days,avg_orders,avg_sales
0,Normal day,1057,88.92,12197.51
1,Holiday,39,63.72,8429.09


In [27]:
df.to_sql("report_holiday_vs_normal", engine, schema="gold", if_exists="replace", index=False)

-1

Satış bayram günündən əvvəl və sonra necə dəyişir?

Bayramdan əvvəlki gün  və bayram günü satış ən aşağıdır. Qalan günlərdə orta sifariş sayı 90.09, satış isə 1234178-dir. Bayramdan sonrakı gün satış bərpa olmağa başlayır, amma hələ də adi günlərdən aşağıdır. Yəni bayram ətrafında satış aktivliyi azalır, bayramdan sonra isə tədricən normala qayıdır.

In [28]:
df = pd.read_sql("""
    WITH holiday_dates AS (
        SELECT sales_date
        FROM gold.mart_daily_sales_weather
        WHERE is_holiday = 1
    ),
    daily_data AS (
        SELECT
            d.total_orders,
            d.total_sales,
            CASE
                WHEN d.is_holiday = 1 THEN 'Holiday'
                WHEN EXISTS (SELECT 1 FROM holiday_dates h
                             WHERE h.sales_date = DATEADD(DAY, 1, d.sales_date))
                     THEN 'Day before holiday'
                WHEN EXISTS (SELECT 1 FROM holiday_dates h
                             WHERE h.sales_date = DATEADD(DAY, -1, d.sales_date))
                     THEN 'Day after holiday'
                ELSE 'Other days'
            END AS day_type
        FROM gold.mart_daily_sales_weather d
    )
    SELECT
        day_type,
        COUNT(*) AS total_days,
        ROUND(AVG(CAST(total_orders AS FLOAT)), 2) AS avg_orders,
        ROUND(AVG(CAST(total_sales AS FLOAT)), 2) AS avg_sales
    FROM daily_data
    GROUP BY day_type
    ORDER BY day_type
""", engine)

df

,day_type,total_days,avg_orders,avg_sales
0,Day after holiday,33,80.70,11005.93
1,Day before holiday,35,63.63,9244.14
2,Holiday,39,63.72,8429.09
3,Other days,989,90.09,12341.78


In [29]:
df.to_sql("report_holiday_around_days", engine, schema="gold", if_exists="replace", index=False)

-1

Adi günlərdə yağışlı günlər yağışsız günlərdən yüksəkdir. Bayram günlərində isə vəziyyət tərsinədir: yağışlı bayram günlərində sifariş sayı və satış ən aşağıdır, yağışsız bayram günlərində isə 75.69 sifariş və 10,721.34 satış olub. yağışlı bayram günlərinin sayı azdır

In [30]:
df = pd.read_sql("""
    SELECT
        day_type,
        COUNT(*) AS total_days,
        ROUND(AVG(CAST(total_orders AS FLOAT)), 2) AS avg_orders,
        ROUND(AVG(CAST(total_sales AS FLOAT)), 2) AS avg_sales
    FROM (
        SELECT
            total_orders,
            total_sales,
            CASE
                WHEN is_holiday = 1 AND sp_precipitation > 0 THEN 'Rainy holiday'
                WHEN is_holiday = 1 THEN 'Dry holiday'
                WHEN sp_precipitation > 0 THEN 'Rainy normal day'
                ELSE 'Dry normal day'
            END AS day_type
        FROM gold.mart_daily_sales_weather
    ) t
    GROUP BY day_type
    ORDER BY day_type
""", engine)

df

,day_type,total_days,avg_orders,avg_sales
0,Dry holiday,13,75.69,10721.34
1,Dry normal day,401,81.97,11361.78
2,Rainy holiday,26,57.73,7282.97
3,Rainy normal day,656,93.18,12708.37


In [31]:
df.to_sql("report_holiday_rain", engine, schema="gold", if_exists="replace", index=False)

-1

Bayram və hava şəraitinin birgə təhlili göstərdiki yağışlı adi günlərdə orta sifariş sayı yağışsız adi günlərdən yüksək olub. Bayram günlərində isə yağışlı hava daha aşağı sifariş sayı və satış gəliri ilə müşahidə edilib. Xüsusilə yağışlı bayram günlərində orta sifariş sayı 57,73, orta satış gəliri isə 7.282,97 olub. Bu nəticələr hava şəraiti və bayram günlərinin satış göstəriciləri ilə əlaqəli ola biləcəyini göstərir.